In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 76 — DSPy RAG Optimizer

## Goal
Compare **hand-written prompts** vs a **DSPy-compiled**
RAG pipeline. Show how DSPy optimizes prompts automatically.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Hand-written RAG prompt** | Traditional prompt engineering |
| **DSPy RAG module** | Declarative retrieval + generation |
| **LabeledFewShot** | Optimize with example demonstrations |
| **Evaluation** | Fair comparison on the same test set |
| **Prompt inspection** | See what DSPy generates under the hood |

## Stack
- `dspy` 3.1+ (module, optimizer, evaluation)
- Ollama `qwen3.5:9b`
- Simple in-memory retriever

In [ ]:
import os, warnings, random
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import dspy

lm = dspy.LM(
    "ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    api_key="fake",
    temperature=0,
)
dspy.configure(lm=lm)
print(f"DSPy {dspy.__version__} configured with Ollama qwen3.5:9b")

## Step 1 — Create Knowledge Base & Retriever

We build a simple keyword-based retriever. In production
you'd use a vector database.

In [ ]:
# Knowledge base documents
KB = [
    {"id": 1, "text": "Python's GIL (Global Interpreter Lock) prevents true multi-threading "
     "for CPU-bound tasks. Use multiprocessing or asyncio for parallelism. The GIL exists "
     "because CPython's memory management is not thread-safe. Libraries like NumPy release "
     "the GIL during computation."},
    {"id": 2, "text": "Docker containers share the host OS kernel, making them lighter than VMs. "
     "Images are built in layers. Each Dockerfile instruction creates a layer. Use multi-stage "
     "builds to reduce image size. Always use .dockerignore to exclude unnecessary files."},
    {"id": 3, "text": "Git branching strategies: GitFlow uses develop/feature/release/hotfix branches. "
     "GitHub Flow is simpler: main + feature branches with PRs. Trunk-Based Development uses "
     "short-lived branches merged frequently into main. Choose based on team size and release cadence."},
    {"id": 4, "text": "SQL indexing: B-tree indexes are default in PostgreSQL/MySQL. They optimize "
     "equality and range queries. Hash indexes are faster for equality-only. Partial indexes "
     "save space by indexing a subset. Always EXPLAIN ANALYZE to check query plans."},
    {"id": 5, "text": "WebSocket provides full-duplex communication between client and server. "
     "Unlike HTTP, the connection stays open. Use cases: chat apps, live dashboards, gaming. "
     "Libraries: Socket.IO (Node.js), websockets (Python), SignalR (.NET). Consider SSE for "
     "server-to-client only streams."},
    {"id": 6, "text": "SOLID principles: Single Responsibility (one reason to change), Open/Closed "
     "(open for extension, closed for modification), Liskov Substitution (subtypes must be "
     "substitutable), Interface Segregation (specific interfaces), Dependency Inversion "
     "(depend on abstractions, not concretions)."},
]

# Simple keyword retriever
def retrieve(query: str, k: int = 2) -> list[str]:
    """Simple keyword-overlap retriever."""
    query_words = set(query.lower().split())
    scored = []
    for doc in KB:
        doc_words = set(doc["text"].lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc["text"]))
    scored.sort(key=lambda x: -x[0])
    return [text for _, text in scored[:k]]

# Test retriever
results = retrieve("What is Python's GIL?")
print(f"Retrieved {len(results)} docs for 'What is Python\'s GIL?':")
print(f"  Doc 1: {results[0][:80]}...")

## Step 2 — Create Q&A Dataset

In [ ]:
# Question-answer pairs
qa_data = [
    {"question": "What is the GIL in Python and why does it exist?",
     "answer": "The GIL is the Global Interpreter Lock that prevents true multi-threading for CPU-bound tasks because CPython's memory management is not thread-safe."},
    {"question": "How are Docker containers different from virtual machines?",
     "answer": "Docker containers share the host OS kernel making them lighter than VMs. Images are built in layers from Dockerfile instructions."},
    {"question": "What are the main Git branching strategies?",
     "answer": "GitFlow uses develop/feature/release/hotfix branches. GitHub Flow uses main plus feature branches with PRs. Trunk-Based Development uses short-lived branches merged frequently."},
    {"question": "How do SQL indexes improve query performance?",
     "answer": "B-tree indexes optimize equality and range queries. Hash indexes are faster for equality-only lookups. Use EXPLAIN ANALYZE to verify query plans."},
    {"question": "When should I use WebSocket vs HTTP?",
     "answer": "WebSocket provides full-duplex persistent connections for real-time communication like chat and live dashboards. Use SSE for server-to-client only streams."},
    {"question": "What are the SOLID principles in software design?",
     "answer": "SOLID stands for Single Responsibility, Open/Closed, Liskov Substitution, Interface Segregation, and Dependency Inversion."},
    {"question": "How do I achieve parallelism in Python despite the GIL?",
     "answer": "Use multiprocessing or asyncio. Libraries like NumPy release the GIL during computation for CPU-bound workloads."},
    {"question": "What are Docker multi-stage builds?",
     "answer": "Multi-stage builds use multiple FROM statements to reduce final image size by copying only needed artifacts from build stages."},
]

examples = [dspy.Example(question=d["question"], answer=d["answer"]).with_inputs("question")
            for d in qa_data]

random.seed(42)
random.shuffle(examples)
trainset = examples[:5]
testset = examples[5:]

print(f"Train: {len(trainset)}, Test: {len(testset)}")

## Step 3 — Hand-Written RAG Prompt

This is the traditional approach: manually craft
the prompt template.

In [ ]:
def handwritten_rag(question: str) -> str:
    """Traditional hand-written RAG prompt."""
    context_docs = retrieve(question, k=2)
    context = "\n".join(f"- {doc}" for doc in context_docs)
    
    prompt = f"""Answer the question based on the provided context.
Be concise and accurate. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}

Answer:"""
    
    response = lm(prompt)
    return response[0]

# Test
ans = handwritten_rag("What is the GIL in Python?")
print(f"Hand-written RAG answer:\n{ans[:300]}")

## Step 4 — DSPy RAG Module

In [ ]:
class RAG(dspy.Module):
    """DSPy RAG module: retrieve context, then generate answer."""
    
    def __init__(self):
        super().__init__()
        self.respond = dspy.ChainOfThought("context, question -> answer")
    
    def forward(self, question):
        context_docs = retrieve(question, k=2)
        context = "\n".join(context_docs)
        return self.respond(context=context, question=question)

rag = RAG()
pred = rag(question="What is the GIL in Python?")
print(f"DSPy RAG answer:\n{pred.answer[:300]}")

## Step 5 — Evaluate & Compare

In [ ]:
# Simple keyword overlap metric
def answer_quality(example, prediction, trace=None):
    """Check keyword overlap between predicted and expected answer."""
    if hasattr(prediction, 'answer'):
        pred_text = prediction.answer.lower()
    else:
        pred_text = str(prediction).lower()
    
    expected_words = set(example.answer.lower().split())
    pred_words = set(pred_text.split())
    
    # Remove common words
    stop = {"the", "a", "an", "is", "are", "was", "were", "of", "for", "to", "in", "and", "or", "it", "that", "this"}
    expected_words -= stop
    pred_words -= stop
    
    if not expected_words:
        return 0.0
    overlap = len(expected_words & pred_words)
    return overlap / len(expected_words)

# Evaluate baseline DSPy RAG
print("Baseline DSPy RAG on test set:")
print("=" * 60)
baseline_scores = []
for ex in testset:
    pred = rag(question=ex.question)
    score = answer_quality(ex, pred)
    baseline_scores.append(score)
    icon = "✅" if score > 0.4 else "⚠️" if score > 0.2 else "❌"
    print(f"  {icon} [{score:.2f}] Q: {ex.question[:50]}...")

avg_baseline = sum(baseline_scores) / len(baseline_scores)
print(f"\nBaseline avg score: {avg_baseline:.2f}")

In [ ]:
# Optimize with LabeledFewShot
optimizer = dspy.LabeledFewShot(k=3)
optimized_rag = optimizer.compile(
    student=RAG(),
    trainset=trainset,
)

# Evaluate optimized
print("Optimized DSPy RAG on test set:")
print("=" * 60)
optimized_scores = []
for ex in testset:
    pred = optimized_rag(question=ex.question)
    score = answer_quality(ex, pred)
    optimized_scores.append(score)
    icon = "✅" if score > 0.4 else "⚠️" if score > 0.2 else "❌"
    print(f"  {icon} [{score:.2f}] Q: {ex.question[:50]}...")

avg_optimized = sum(optimized_scores) / len(optimized_scores)
print(f"\nOptimized avg score: {avg_optimized:.2f}")

In [ ]:
# Also evaluate the hand-written approach
print("Hand-written RAG on test set:")
print("=" * 60)
hw_scores = []
for ex in testset:
    ans = handwritten_rag(ex.question)
    pred_obj = dspy.Prediction(answer=ans)
    score = answer_quality(ex, pred_obj)
    hw_scores.append(score)
    icon = "✅" if score > 0.4 else "⚠️" if score > 0.2 else "❌"
    print(f"  {icon} [{score:.2f}] Q: {ex.question[:50]}...")

avg_hw = sum(hw_scores) / len(hw_scores)

print(f"\n{'='*60}")
print(f"FINAL COMPARISON")
print(f"{'='*60}")
print(f"  Hand-written RAG:    {avg_hw:.2f}")
print(f"  DSPy RAG (baseline): {avg_baseline:.2f}")
print(f"  DSPy RAG (optimized): {avg_optimized:.2f}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Hand-written prompt** | Manual f-string template |
| **DSPy Module** | `class RAG(dspy.Module)` with `forward()` |
| **ChainOfThought** | Adds reasoning before answering |
| **LabeledFewShot** | Auto-selects training examples for the prompt |
| **Metric** | Custom function scoring prediction quality |

## Hand-Written vs DSPy
```
Hand-Written                    DSPy
──────────                      ────
Manual prompt engineering       Declarative signatures
Fixed template                  Auto-optimized prompts
Trial and error                 Systematic optimization
Hard to iterate                 Easy to swap strategies
Full control                    Less control, more automation
```

## Extending This Notebook
- Try `dspy.BootstrapFewShot` (generates synthetic demonstrations)
- Try `dspy.MIPROv2` (optimizes both instructions and demos)
- Replace keyword retriever with a vector-based one
- Add `dspy.Evaluate` for parallelized evaluation
- Save optimized program with `optimized_rag.save('rag.json')`